# Proyecto Dashboard Stock

#### Contexto del Proyecto

In [68]:

# ---------------------------------------------------------
# Este archivo sirve como base lógica para migrar el sistema de fórmulas de Google Sheets a un entorno de Python

# Por medio de formularios de google forms se registra recepción, apertura y eventos especiales de antibióticos y de insumos.
# Las hojas de google sheet son StockAntibióticos y StockInsumos, cada fila corresponde a una respuesta de un formulario y
# cada columna corresponde a una pregunta del formulario.

# Los antibióticos, se reciben en 2 presentaciones: E-Test y sensidiscos. 
# Para conveniencia, las cajas de E-test se registran de a 1 (25 unidades aprox), mientras que en cada caja de sensidisco vienen 
# 5 tubos con 50 discos cada uno, por lo que se registra cada tubo como una unidad (50 discos) cada caja son 5 unidades de sensidiscos.


# 2. OBJETIVOS TÉCNICOS:
#   - Conexión vía API (gspread) y manipulación de estructuras de datos.
#   - Limpieza de fechas de apertura/vencimiento y cálculo de stock neto.
#   - Predicción de quiebre de stock basado en consumo histórico y fechas de vencimiento.

#-- LÓGICA DE EVENTOS ---
 #   Existen 3 tipos de eventos:
 #       Evento por quiebre de stock.
 #       Evento por control de calidad fallido
 #       Evento por vencimiento.
 #   Los 2 primeros funcionan solo como registro del evento.
#    El evento por vencimiento también indica cuántas cantidades fueron eliminadas por caducadas.

#--- LÓGICA DE CÁLCULO CORRECTA ---
#    La lógica para calcular la duración de cada unidad es calculando el tiempo entre 2 aperturas de un mismo item siempre y cuando
#    no haya un evento de ese mismo item entre 2 aperturas consecutivas.

# Antibióticos como Herramientas de Diagnóstico
# En un laboratorio de microbiología, los antibióticos (en formato de sensidiscos o E-test) no son fármacos terapéuticos, 
# sino reactivos de diagnóstico crítico. Su función no es curar al paciente directamente, sino actuar como sensores biológicos 
# para determinar el perfil de susceptibilidad de un patógeno.  

# Impacto de la Ruptura de Stock: Si la hoja de StockAntibióticos muestra un quiebre de stock (stock = 0), 
# la consecuencia no es la falta de un tratamiento, sino la generación de un vacío de información diagnóstica.  

#Consecuencia Clínica: Sin el insumo para el testeo, el equipo médico queda "a ciegas", perdiendo la capacidad de
# realizar una terapia dirigida y viéndose obligado a mantener esquemas empíricos de amplio espectro, lo que aumenta 
# la presión selectiva y la resistencia bacteriana.  

# El proyecto busca precisamente evitar este escenario mediante la predicción de demanda,
# asegurando que la capacidad diagnóstica del hospital nunca se vea interrumpida.



#### Importaciones y Conexión a Google Sheets

In [69]:
import gspread
import pandas as pd
import numpy as np
from google.oauth2.service_account import Credentials
from datetime import datetime, timedelta

scopes = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive",
]
creds = Credentials.from_service_account_file("credentials.json", scopes=scopes)

gc = gspread.authorize(creds)

# Open the Google Sheet by name or URL
sheet = gc.open("Registro de antibióticos (respuestas)")

# Para tener las dos hojas de trabajo:
worksheet_atb = sheet.worksheet("StockAntibióticos")
worksheet_ins = sheet.worksheet("StockInsumos")

# Leer toda la data de worksheet_atb y worksheet_ins
data_atb = worksheet_atb.get_all_values()
data_ins = worksheet_ins.get_all_values()   

# Authorize gspread
gc = gspread.authorize(creds)

# Configurar el ancho total de la pantalla (en caracteres)
# Aumenta este valor (p.ej., 200 o 300) según el tamaño de tu monitor
pd.set_option('display.width', 1500)

# crear dataframe de pandas a partir de los datos obtenidos
df_atb = pd.DataFrame(data_atb[1:], columns=data_atb[0])
df_ins = pd.DataFrame(data_ins[1:], columns=data_ins[0])

df_atb = df_atb.rename(columns={'Antibiótico': 'nombre_item'})
df_ins = df_ins.rename(columns={'Insumo': 'nombre_item'})

# 2. Configurar el ancho total de la pantalla (en caracteres)
# Aumenta este valor (p.ej., 200 o 300) según el tamaño de tu monitor
pd.set_option('display.width', 2000)

#### Exploración y Selección de Datos

In [70]:
dayfirst=True
# Exploración de los datos: 
# print(df.shape)
#print(df.columns)
#print(df.head())

#para comprobar el tipo de dato de cada columna, podemos usar el siguiente código:
# print(df.dtypes)

#vamos a mostrar el nombre de las columnas para tener una referencia clara de cómo se llaman:
# print(df.columns)

# Creamos categoría de item para cada dataframe, esto nos servirá para luego poder diferenciar entre antibióticos e insumos en el análisis,
#  ya que ambos dataframes tienen la misma estructura pero corresponden a categorías diferentes de productos.

df_atb['categoria'] = 'Antibiótico'
df_ins['categoria'] = 'Insumo'

df_atb_ins = pd.concat([df_atb, df_ins])

# Display the data
# print(df_atb_ins)

#vamos a renombrar las columnas para que tengan nombres más amigables y fáciles de usar para ambos dataframes, ya que ambos tienen las mismas columnas pero con nombres diferentes
mapeo_columnas = {
    'Marca temporal': 'marca_temporal',
    'Tipo de registro': 'tipo_registro',
        'Fecha de recepción': 'fecha_recepcion',
    'Fecha de apertura': 'fecha_apertura',
    'Fecha de evento': 'fecha_evento',
    'Fecha de vencimiento': 'fecha_vencimiento',
    'Cantidad': 'cantidad_recibida',
    'Cantidad eliminada': 'cantidad_eliminada',
    'Responsable': 'responsable',
    'Comentario': 'comentario'
    }

#renombramos las columnas de ambos dataframe usando el mapeo definido anteriormente:
df_atb_ins = df_atb_ins.rename(columns=mapeo_columnas)

# Al principio los Eventos se registraban como Cierres, entonces para unificarlos usamos:
df_atb_ins['tipo_registro'] = df_atb_ins['tipo_registro'].replace({'Cierre': 'Evento'})

# para comprobar que se renombraron correctamente las columnas, podemos imprimir las columnas nuevamente:
# print("columnas df_atb_ins:", df_atb_ins.columns)


#Eliminaremos las columnas que no nos son útiles para el análisis, como
# timestamp, responsable, comentario, ya que no aportan información relevante para el cálculo de stock y predicción de quiebre.
columnas_a_eliminar = ['responsable', 'timestamp', 'comentario']
df_atb_ins = df_atb_ins.drop(columns=columnas_a_eliminar, errors='ignore')
# para comprobar que se eliminaron las columnas correctamente, podemos imprimir las columnas nuevamente:
# print(df_atb_ins.head())

# para convertir las columnas a su formato correcto
df_atb_ins['cantidad_recibida'] = pd.to_numeric(df_atb_ins['cantidad_recibida'], errors='coerce').astype('Int64')
df_atb_ins['cantidad_eliminada'] = pd.to_numeric(df_atb_ins['cantidad_eliminada'], errors='coerce').astype('Int64')
columnas_fecha = ['fecha_recepcion', 'fecha_apertura', 'fecha_evento', 'fecha_vencimiento']
for col in columnas_fecha:
    df_atb_ins[col] = pd.to_datetime(df_atb_ins[col], dayfirst=True, errors='coerce')

# para comprobar que las columnas de fechas se convirtieron correctamente, podemos imprimir el tipo de dato de cada columna nuevamente:
# print("df_atb_ins.dtypes:", df_atb_ins.dtypes)

#Identificamos las aperturas y les asignamos el valor 1 (descuento)
df_atb_ins['unidad_consumida'] = df_atb_ins['tipo_registro'].apply(lambda x: 1 if x == 'Apertura' else 0).astype('Int64')  

# para que la columna de categoría quede antes de la columna nombre_item:
cols = df_atb_ins.columns.tolist()
cols.insert(cols.index('nombre_item'), cols.pop(cols.index('categoria')))
df_atb_ins = df_atb_ins[cols]   

# Eliminar índices duplicados (vital después de un concat)
df_atb_ins = df_atb_ins.reset_index(drop=True)

print("df_atb_ins", df_atb_ins)


df_atb_ins           marca_temporal tipo_registro    categoria                               nombre_item fecha_recepcion  cantidad_recibida fecha_apertura fecha_evento                      Evento fecha_vencimiento  cantidad_eliminada  unidad_consumida
0      6/01/2026 8:02:24      Apertura  Antibiótico                    Aztreonam (Sensidisco)             NaT               <NA>     2025-08-29          NaT                                           NaT                <NA>                 1
1      6/01/2026 8:02:39      Apertura  Antibiótico                    Amikacina (Sensidisco)             NaT               <NA>     2025-08-29          NaT                                           NaT                <NA>                 1
2      6/01/2026 8:02:56      Apertura  Antibiótico               Ciprofloxacino (Sensidisco)             NaT               <NA>     2025-12-09          NaT                                           NaT                <NA>                 1
3      6/01/2026 8:03:19 

#### Dashboard

##### Creación DataFrame

In [71]:

# Columna F: Consumo promedio en días por unidad (Lógica: Tiempo entre aperturas / Cantidad abierta) siempre que no haya eventos entre aperturas
# Columna G: Alertas de Vencimiento (Lógica: Si fecha_vencimiento - fecha_actual <= 30 días, entonces "Alerta", sino "OK")
# Columna H: Predicción de Quiebre de Stock (Lógica: Modelo de ML basado en consumo histórico y fechas de vencimiento)
# Columna I: Alertas de Consumo (Lógica: Si consumo promedio en días por unidad <= 7 días, entonces "Alerta", sino "OK")
# Columna J: Recomendación de Compra (Lógica: Si Stock Disponible <= 10 unidades o Predicción de Quiebre de Stock es "Alerta", entonces "Comprar", sino "No Comprar")
# Columna K: Historial de Consumo (Lógica: Gráfico de consumo histórico por antibiótico)
# Columna L: Historial de Eventos (Lógica: Lista de eventos registrados por antibiótico)
# Columna M: Historial de Vencimientos (Lógica: Lista de fechas de vencimiento por antibiótico)
# Columna N: Historial de Recepciones (Lógica: Lista de fechas y cantidades de recepciones por antibiótico)
# Columna O: Historial de Aperturas (Lógica: Lista de fechas de aperturas por antibiótico)
# Columna P: Historial de Eliminaciones (Lógica: Lista de fechas y cantidades de eliminaciones por antibiótico)
# para concatenar ambos dataframes, podemos usar la función pd.concat() de pandas, pero antes debemos asegurarnos de que ambos dataframes tengan las mismas columnas y el mismo formato, para luego aplicar las agregaciones necesarias para cada columna del dashboard.

# df_dashboard.describe()
# print (df_atb_ins.head())
# print (df_atb_ins.tail())


# Asegurar que cantidad_eliminada sea numérica (Int64 para manejar <NA>)

df_atb_ins['cantidad_eliminada'] = pd.to_numeric(df_atb_ins['cantidad_eliminada'], errors='coerce').astype('Int64')

# print (df_atb_ins)
# Crear el dashboard consolidado usando pivot_table (más seguro que lambda)
df_pivot = df_atb_ins.pivot_table(
    index='nombre_item',
    columns='tipo_registro',
    values=['cantidad_recibida', 'unidad_consumida', 'cantidad_eliminada'],
    aggfunc='sum',
    fill_value=0 # Importante para no heredar valores de otros ítems [cite: 61]
).reset_index()

#print (df_pivot)
# Aplanar y limpiar nombres (evita columnas duplicadas como 'cantidad_eliminada_Apertura')
df_pivot.columns = ['_'.join(col).strip('_') for col in df_pivot.columns.values]
#print (df_pivot)

# 
df_dashboard = pd.DataFrame()
df_dashboard['nombre_item'] = df_pivot['nombre_item']

##### Columnas B, C, D y E

In [72]:

# Columna B: Recuento de Recibidos
df_dashboard['Recibido'] = df_pivot.get('cantidad_recibida_Recepción', 0)

# Columna C: Recuento de Consumidos 
df_dashboard['Consumido'] = df_pivot.get('unidad_consumida_Apertura', 0)

# Columna D: Recuento de Descartes (Lógica: Suma de cantidades eliminadas por eventos de tipo "Cierre" o "Evento")
df_dashboard['Descartes'] = (df_pivot.get('cantidad_eliminada_Evento', 0))

# Columna E: Cálculo de Stock Disponible (Lógica: Recibido - Abierto - Eliminado)
df_dashboard['Stock actual'] = (
    df_dashboard['Recibido'].fillna(0) - 
    (df_dashboard['Consumido'].fillna(0) + df_dashboard['Descartes'].fillna(0))
)

print (df_dashboard)


                                 nombre_item  Recibido  Consumido  Descartes  Stock actual
0                         Amikacina (E-test)         1          1          0             0
1                     Amikacina (Sensidisco)        10          5          0             5
2   Amoxicilina/Ac. clavulánico (Sensidisco)         9          1          3             5
3                    Ampicilina (Sensidisco)         5          2          0             3
4          Ampicilina/Sulbactam (Sensidisco)         4          1          0             3
5                      Azitromicina (E-test)         1          1          0             0
6                  Azitromicina (Sensidisco)         5          3          0             2
7                         Aztreonam (E-test)         2          1          0             1
8                     Aztreonam (Sensidisco)         7          2          0             5
9                                      CORIS         4          1          0             3

##### Calc_logica aperturas

In [73]:

# ---- CALC_LOGICA ----
# Filtrar solo los eventos relevantes para el consumo y ordenar cronológicamente
df_calc_logica = df_atb_ins[df_atb_ins['tipo_registro'].isin(['Apertura', 'Evento'])].copy()
pd.set_option('display.max_rows', None)

#Para eliminar las columnas que no nos interesan para el calculo de la lógica:
columnas_a_eliminar = ['marca_temporal', 'fecha_recepcion', 'cantidad_recibida', 'fecha_vencimiento', 'Evento', 'cantidad_eliminada', 'unidad_consumida', 'categoría']
df_calc_logica = df_atb_ins.drop(columns=columnas_a_eliminar, errors='ignore')
df_calc_logica['fecha_calculo'] = np.where(
    df_calc_logica['tipo_registro'] == 'Apertura', 
    df_calc_logica['fecha_apertura'], 
    df_calc_logica['fecha_evento'])

df_calc_logica = df_calc_logica[df_calc_logica['tipo_registro'] != 'Recepción']
df_calc_logica = df_calc_logica.sort_values(by=['nombre_item', 'fecha_calculo'])

# Identificar el tipo y la fecha del siguiente registro para comparar
df_calc_logica['sig_tipo'] = df_calc_logica.groupby('nombre_item')['tipo_registro'].shift(-1)
df_calc_logica['sig_fecha'] = df_calc_logica.groupby('nombre_item')['fecha_calculo'].shift(-1)

# Calcular la diferencia de días entre eventos
df_calc_logica['dias_ciclo'] = (df_calc_logica['sig_fecha'] - df_calc_logica['fecha_calculo']).dt.days

# Aplicar la validación: Solo Apertura -> Apertura es válido
# Si hay un 'Evento' (Descarte/Falla) a continuación, se ignora el ciclo
df_calc_logica['consumo_prom_dias'] = df_calc_logica.apply(
    lambda row: row['dias_ciclo'] if (row['tipo_registro'] == 'Apertura' and row['sig_tipo'] == 'Apertura') 
    else pd.NA, axis=1)

# Convertir a Int64 para mantener la consistencia con celdas <NA>
df_calc_logica['consumo_prom_dias'] = df_calc_logica['consumo_prom_dias'].astype('Int64')



# ---- FIN CALC_LOGICA ----

print (df_calc_logica)


    tipo_registro    categoria                               nombre_item fecha_apertura fecha_evento fecha_calculo  sig_tipo  sig_fecha  dias_ciclo  consumo_prom_dias
148      Apertura  Antibiótico                        Amikacina (E-test)     2026-03-11          NaT    2026-03-11       NaN        NaT         NaN               <NA>
1        Apertura  Antibiótico                    Amikacina (Sensidisco)     2025-08-29          NaT    2025-08-29  Apertura 2026-02-05       160.0                160
105      Apertura  Antibiótico                    Amikacina (Sensidisco)     2026-02-05          NaT    2026-02-05  Apertura 2026-03-05        28.0                 28
127      Apertura  Antibiótico                    Amikacina (Sensidisco)     2026-03-05          NaT    2026-03-05  Apertura 2026-03-26        21.0                 21
165      Apertura  Antibiótico                    Amikacina (Sensidisco)     2026-03-26          NaT    2026-03-26    Evento 2026-04-17        22.0               <NA

##### Columna F: Consumo promedio

In [74]:

# Columna F: Consumo promedio en días por unidad 
# Lógica: Tiempo entre aperturas / Cantidad abierta) siempre que no haya eventos entre aperturas
# Calculamos el promedio de días por cada producto
# Pandas ignorará automáticamente los valores <NA> o NaN en el cálculo del mean()
df_promedios = df_calc_logica.groupby('nombre_item')['consumo_prom_dias'].mean().reset_index().round(0)

# print (df_promedios)
# Renombramos la columna para que sea clara en el Dashboard
# df_promedios = df_promedios.rename(columns={'consumo_prom_dias': 'consumo_promedio'})

# Unimos al dashboard asegurando que coincidan los nombres
df_dashboard = pd.merge(df_dashboard, df_promedios, on='nombre_item', how='left')

print (df_dashboard)



                                 nombre_item  Recibido  Consumido  Descartes  Stock actual  consumo_prom_dias
0                         Amikacina (E-test)         1          1          0             0               <NA>
1                     Amikacina (Sensidisco)        10          5          0             5               70.0
2   Amoxicilina/Ac. clavulánico (Sensidisco)         9          1          3             5               <NA>
3                    Ampicilina (Sensidisco)         5          2          0             3               83.0
4          Ampicilina/Sulbactam (Sensidisco)         4          1          0             3               <NA>
5                      Azitromicina (E-test)         1          1          0             0               <NA>
6                  Azitromicina (Sensidisco)         5          3          0             2              106.0
7                         Aztreonam (E-test)         2          1          0             1               <NA>
8         

##### Columna G: Días de autonomía

In [75]:

# Columna G:  Calcular Días de Autonomía (Días de Stock Disponible)
# Lógica: (stock_actual + 1 )* consumo_prom_dias
# Usamos un manejo de errores para evitar divisiones por cero o valores nulos
df_dashboard['dias_autonomia'] = (
    (df_dashboard['Stock actual'] + 1) * df_dashboard['consumo_prom_dias']
).replace([float('inf'), -float('inf')], pd.NA)

print (df_dashboard)


                                 nombre_item  Recibido  Consumido  Descartes  Stock actual  consumo_prom_dias  dias_autonomia
0                         Amikacina (E-test)         1          1          0             0               <NA>            <NA>
1                     Amikacina (Sensidisco)        10          5          0             5               70.0           420.0
2   Amoxicilina/Ac. clavulánico (Sensidisco)         9          1          3             5               <NA>            <NA>
3                    Ampicilina (Sensidisco)         5          2          0             3               83.0           332.0
4          Ampicilina/Sulbactam (Sensidisco)         4          1          0             3               <NA>            <NA>
5                      Azitromicina (E-test)         1          1          0             0               <NA>            <NA>
6                  Azitromicina (Sensidisco)         5          3          0             2              106.0         

##### Cálculo última apertura

In [76]:
#  ---- CALCULO DE ULTIMA APERTURA ----
# Creamos un DataFrame auxiliar con las aperturas de cada ítem, ya que para calcular la última fecha de apertura 
# necesitamos filtrar solo las filas que correspondan a aperturas.
df_aperturas = df_atb_ins[df_atb_ins['tipo_registro'] == 'Apertura']
df_aperturas = df_aperturas.sort_values(by=['nombre_item', 'fecha_apertura'])

# Calculamos la última fecha de apertura por cada item
# Usamos transform('max') para que el resultado mantenga el mismo número de filas
df_aperturas['ultima_apertura'] = df_aperturas.groupby('nombre_item')['fecha_apertura'].transform('max')

# Agrupamos por nombre de ítem y seleccionamos el máximo de ambas columnas
# Esto nos dará el último registro de apertura y el valor de la última apertura calculada
df_ultima_apertura = df_aperturas.groupby('nombre_item').agg({
    'fecha_apertura': 'max',
    'ultima_apertura': 'first' # Tomamos el primero porque todos son iguales tras el cálculo
}).reset_index()

print(df_ultima_apertura)

                                 nombre_item fecha_apertura ultima_apertura
0                         Amikacina (E-test)     2026-03-11      2026-03-11
1                     Amikacina (Sensidisco)     2026-04-17      2026-04-17
2   Amoxicilina/Ac. clavulánico (Sensidisco)     2026-01-08      2026-01-08
3                    Ampicilina (Sensidisco)     2026-03-17      2026-03-17
4          Ampicilina/Sulbactam (Sensidisco)     2026-02-18      2026-02-18
5                      Azitromicina (E-test)     2026-02-12      2026-02-12
6                  Azitromicina (Sensidisco)     2026-04-14      2026-04-14
7                         Aztreonam (E-test)     2026-02-02      2026-02-02
8                     Aztreonam (Sensidisco)     2026-02-19      2026-02-19
9                                      CORIS     2026-04-21      2026-04-21
10                  Cefadroxilo (Sensidisco)     2025-12-22      2025-12-22
11                     Cefepime (Sensidisco)     2026-03-02      2026-03-02
12          

##### Columna H: Fecha de quiebre

In [77]:
# Columna H: Predicción fecha de quiebre de stock
# Unir este valor al df_dashboard principal
# df_dashboard = pd.merge(df_dashboard, df_ultima_apertura[['nombre_item', 'ultima_apertura']], on='nombre_item', how='left')
# Unimos los dataframes por la columna del nombre del ítem
# 'left' asegura que mantengamos todos los ítems de nuestro dashboard principal
df_dashboard = pd.merge(df_dashboard, 
                        df_ultima_apertura[['nombre_item', 'ultima_apertura']], 
                        on='nombre_item', 
                        how='left')

# 2. Ahora que están en la misma tabla, la suma es directa y segura
df_dashboard['fecha_quiebre'] = df_dashboard['ultima_apertura'] + pd.to_timedelta(df_dashboard['dias_autonomia'], unit='d')
# Ahora se puede usar 'ultima_apertura' para el cálculo de quiebre multiplicativo:

print(df_dashboard)
# 3. Formatear para visualización en Streamlit (Opcional)
# df_dashboard['fecha_quiebre_display'] = df_dashboard['fecha_quiebre'].dt.strftime('%d/%m/%Y')


                                 nombre_item  Recibido  Consumido  Descartes  Stock actual  consumo_prom_dias  dias_autonomia ultima_apertura fecha_quiebre
0                         Amikacina (E-test)         1          1          0             0               <NA>            <NA>      2026-03-11           NaT
1                     Amikacina (Sensidisco)        10          5          0             5               70.0           420.0      2026-04-17    2027-06-11
2   Amoxicilina/Ac. clavulánico (Sensidisco)         9          1          3             5               <NA>            <NA>      2026-01-08           NaT
3                    Ampicilina (Sensidisco)         5          2          0             3               83.0           332.0      2026-03-17    2027-02-12
4          Ampicilina/Sulbactam (Sensidisco)         4          1          0             3               <NA>            <NA>      2026-02-18           NaT
5                      Azitromicina (E-test)         1          

##### Columna I: Alertas de Consumo

In [78]:
# Columna I: Alertas de Consumo
# 
# para que se muestre Bajo entre 61 y 90, Alerta entre 1 y 60, y OK para cuando hayan más de 90 días hasta la fecha_quiebre:
# ultima_apertura + dias_autonomia <= hoy + 90 días -> Alerta, <= hoy + 60 días -> Bajo, > hoy + 90 días -> OK, lo demás sin stock 
hoy = pd.to_datetime('today')
df_dashboard['alerta_consumo'] = df_dashboard.apply(
    lambda row: 'Sin datos' if pd.isna(row['fecha_quiebre']) or pd.isna(row['consumo_prom_dias']) else (
        'Sin Stock' if (row['fecha_quiebre'] < hoy) else (
            'Alerta' if row['fecha_quiebre'] <= hoy + pd.Timedelta(days=60) else (
                'Bajo' if row['fecha_quiebre'] <= hoy + pd.Timedelta(days=90) else 'OK'
            )
        )
    ), axis=1
)

print (df_dashboard)

                                 nombre_item  Recibido  Consumido  Descartes  Stock actual  consumo_prom_dias  dias_autonomia ultima_apertura fecha_quiebre alerta_consumo
0                         Amikacina (E-test)         1          1          0             0               <NA>            <NA>      2026-03-11           NaT      Sin datos
1                     Amikacina (Sensidisco)        10          5          0             5               70.0           420.0      2026-04-17    2027-06-11             OK
2   Amoxicilina/Ac. clavulánico (Sensidisco)         9          1          3             5               <NA>            <NA>      2026-01-08           NaT      Sin datos
3                    Ampicilina (Sensidisco)         5          2          0             3               83.0           332.0      2026-03-17    2027-02-12             OK
4          Ampicilina/Sulbactam (Sensidisco)         4          1          0             3               <NA>            <NA>      2026-02-18    